# UpSet Plot: CRISPR Groups vs Differentially Expressed Gene Groups

**Purpose**: Visualize the overlap between CRISPR screening groups (Depleted/Enriched) and RNA-seq differential expression groups (Up/Down/No_change).

**Input file**: `crispr_rnaseq_merged.csv`

**Output**:
- UpSet plot images
- Cross-tabulation table and heatmap
- Gene lists for key combinations

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# UpSet plot library
try:
    from upsetplot import plot as upset_plot
    from upsetplot import UpSet
    print('✓ upsetplot library loaded')
except ImportError:
    print('⚠ upsetplot not installed. Run: pip install upsetplot')
    raise

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('✓ imports ready')


In [ ]:
# === Parameters ===
INPUT_CSV = '/path/to/CRISPR/results/crispr_rnaseq_merged.csv'
RESULTS_DIR = '/path/to/CRISPR/results'
OUTPUT_PREFIX = 'UpSet_CRISPR_RNAseq'

# Column names
COL_GENE_ID = 'gene_id'
COL_SYMBOL = 'gene_symbol'
COL_CRISPR_CATEGORY = 'crispr_category'
COL_RNA_STATUS = 'rna_deg_status'

print('✓ Parameters set')


In [ ]:
# === Load data ===
df = pd.read_csv(INPUT_CSV)
print(f'Loaded {len(df)} genes')
print(f'\nColumns: {list(df.columns)}')
print(f'\nData preview:')
display(df.head())

# Check required columns
required_cols = [COL_GENE_ID, COL_SYMBOL, COL_CRISPR_CATEGORY, COL_RNA_STATUS]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

print('\n✓ Data loaded successfully')


In [ ]:
# === Prepare data for UpSet plot ===

# Count genes in each group
print('=== CRISPR Group Statistics ===')
print(df[COL_CRISPR_CATEGORY].value_counts().sort_index())

print('\n=== RNA-seq Differential Expression Statistics ===')
print(df[COL_RNA_STATUS].value_counts().sort_index())

# Create boolean columns for UpSet plot
# CRISPR groups
df['CRISPR_Depleted'] = df[COL_CRISPR_CATEGORY] == 'Depleted'
df['CRISPR_Slightly_Enriched'] = df[COL_CRISPR_CATEGORY] == 'Slightly Enriched'
df['CRISPR_Moderately_Enriched'] = df[COL_CRISPR_CATEGORY] == 'Moderately Enriched'
df['CRISPR_Highly_Enriched'] = df[COL_CRISPR_CATEGORY] == 'Highly Enriched'

# RNA-seq differential expression groups
df['RNA_Up'] = df[COL_RNA_STATUS] == 'Up'
df['RNA_Down'] = df[COL_RNA_STATUS] == 'Down'
df['RNA_No_change'] = df[COL_RNA_STATUS] == 'No_change'

print('\n✓ Boolean columns created for UpSet plot')


In [ ]:
# === UpSet Plot: CRISPR (4 groups) vs RNA-seq (3 groups) ===
print('=== UpSet Plot: CRISPR Categories vs RNA-seq DEG Status ===')

# Select columns for UpSet plot (order: CRISPR first, RNA-seq second)
upset_cols = [
    'CRISPR_Highly_Enriched',
    'CRISPR_Moderately_Enriched',
    'CRISPR_Slightly_Enriched',
    'CRISPR_Depleted',
    'RNA_Up',
    'RNA_Down',
    'RNA_No_change'
]

# Prepare data: set MultiIndex
upset_data = df.set_index(upset_cols)[COL_SYMBOL]

# Plot UpSet (specify category order, no automatic sorting)
fig = plt.figure(figsize=(14, 8))
upset = UpSet(
    upset_data,
    subset_size='count',
    intersection_plot_elements=10,
    show_counts=True,
    sort_by='cardinality',
    sort_categories_by=None  # Keep list order: CRISPR first, RNA-seq second
)
upset.plot(fig=fig)
plt.suptitle('UpSet Plot: CRISPR Categories vs RNA-seq DEG Status', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print('\n✓ UpSet Plot complete')


In [ ]:
# === Cross-Tabulation Table ===
print('=== Cross-Tabulation: CRISPR Groups vs RNA-seq Groups ===')

# Create cross-tabulation
crosstab = pd.crosstab(
    df[COL_CRISPR_CATEGORY],
    df[COL_RNA_STATUS],
    margins=True,
    margins_name='Total'
)

print('\nGene counts:')
display(crosstab)

# Percentage table
crosstab_pct = pd.crosstab(
    df[COL_CRISPR_CATEGORY],
    df[COL_RNA_STATUS],
    normalize='index'
) * 100

print('\nPercentage (row percentage):')
display(crosstab_pct.round(1))

# Save cross-tabulation
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
crosstab_file = f'{RESULTS_DIR}/{OUTPUT_PREFIX}_crosstab.csv'
crosstab.to_csv(crosstab_file)
print(f'\n✓ Cross-tabulation saved: {crosstab_file}')


In [ ]:
# === Heatmap Visualization of Cross-Tabulation ===
print('=== Heatmap: CRISPR Groups vs RNA-seq Groups ===')

# Plot heatmap (gene counts + percentages)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Gene counts
sns.heatmap(
    crosstab.iloc[:-1, :-1],  # Exclude Total row and column
    annot=True,
    fmt='d',
    cmap='YlOrRd',
    cbar_kws={'label': 'Gene Count'},
    ax=axes[0]
)
axes[0].set_title('Gene Counts', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('RNA-seq DEG Status', fontsize=12, fontweight='bold')
axes[0].set_ylabel('CRISPR Category', fontsize=12, fontweight='bold')

# Right panel: Percentages
sns.heatmap(
    crosstab_pct,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    cbar_kws={'label': 'Percentage (%)'},
    ax=axes[1]
)
axes[1].set_title('Percentage (Row %)', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('RNA-seq DEG Status', fontsize=12, fontweight='bold')
axes[1].set_ylabel('CRISPR Category', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n✓ Heatmap complete')


In [ ]:
# === Export Key Gene Combinations ===
print('=== Export Key Gene Combinations ===')

# 1. CRISPR Highly Enriched + RNA Up
group1 = df[
    (df[COL_CRISPR_CATEGORY] == 'Highly Enriched') & 
    (df[COL_RNA_STATUS] == 'Up')
][[COL_GENE_ID, COL_SYMBOL, 'crispr_score', 'rna_log2fc', 'rna_padj']]
print(f'\n1. CRISPR Highly Enriched + RNA Up: {len(group1)} genes')
display(group1)
group1.to_csv(f'{RESULTS_DIR}/{OUTPUT_PREFIX}_HighEnriched_Up.csv', index=False)

# 2. CRISPR Highly Enriched + RNA Down
group2 = df[
    (df[COL_CRISPR_CATEGORY] == 'Highly Enriched') & 
    (df[COL_RNA_STATUS] == 'Down')
][[COL_GENE_ID, COL_SYMBOL, 'crispr_score', 'rna_log2fc', 'rna_padj']]
print(f'\n2. CRISPR Highly Enriched + RNA Down: {len(group2)} genes')
display(group2)
group2.to_csv(f'{RESULTS_DIR}/{OUTPUT_PREFIX}_HighEnriched_Down.csv', index=False)

# 3. CRISPR Depleted + RNA Up
group3 = df[
    (df[COL_CRISPR_CATEGORY] == 'Depleted') & 
    (df[COL_RNA_STATUS] == 'Up')
][[COL_GENE_ID, COL_SYMBOL, 'crispr_score', 'rna_log2fc', 'rna_padj']]
print(f'\n3. CRISPR Depleted + RNA Up: {len(group3)} genes')
display(group3)
group3.to_csv(f'{RESULTS_DIR}/{OUTPUT_PREFIX}_Depleted_Up.csv', index=False)

# 4. CRISPR Depleted + RNA Down
group4 = df[
    (df[COL_CRISPR_CATEGORY] == 'Depleted') & 
    (df[COL_RNA_STATUS] == 'Down')
][[COL_GENE_ID, COL_SYMBOL, 'crispr_score', 'rna_log2fc', 'rna_padj']]
print(f'\n4. CRISPR Depleted + RNA Down: {len(group4)} genes')
display(group4)
group4.to_csv(f'{RESULTS_DIR}/{OUTPUT_PREFIX}_Depleted_Down.csv', index=False)

print('\n✓ Key gene combinations exported')


In [ ]:
# === Save All Figures ===
print('=== Saving Figures ===')

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# UpSet Plot (keep category order: CRISPR first, RNA-seq second)
fig = plt.figure(figsize=(14, 8))
upset_data = df.set_index(upset_cols)[COL_SYMBOL]
upset = UpSet(upset_data, subset_size='count', show_counts=True, 
              sort_by='cardinality', sort_categories_by=None)
upset.plot(fig=fig)
plt.suptitle('UpSet Plot: CRISPR Categories vs RNA-seq DEG Status', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/{OUTPUT_PREFIX}_upset.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{RESULTS_DIR}/{OUTPUT_PREFIX}_upset.pdf', bbox_inches='tight')
plt.close(fig)
print(f'✓ Saved: {OUTPUT_PREFIX}_upset.png/pdf')

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(crosstab.iloc[:-1, :-1], annot=True, fmt='d', cmap='YlOrRd',
            cbar_kws={'label': 'Gene Count'}, ax=axes[0])
axes[0].set_title('Gene Counts', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('RNA-seq DEG Status', fontsize=12, fontweight='bold')
axes[0].set_ylabel('CRISPR Category', fontsize=12, fontweight='bold')

sns.heatmap(crosstab_pct, annot=True, fmt='.1f', cmap='Blues',
            cbar_kws={'label': 'Percentage (%)'}, ax=axes[1])
axes[1].set_title('Percentage (Row %)', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('RNA-seq DEG Status', fontsize=12, fontweight='bold')
axes[1].set_ylabel('CRISPR Category', fontsize=12, fontweight='bold')

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/{OUTPUT_PREFIX}_heatmap.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{RESULTS_DIR}/{OUTPUT_PREFIX}_heatmap.pdf', bbox_inches='tight')
plt.close(fig)
print(f'✓ Saved: {OUTPUT_PREFIX}_heatmap.png/pdf')

print('\n✓ All figures saved')


In [ ]:
# === Summary ===
print('=== Analysis Summary ===')
print(f'\nTotal genes: {len(df)}')
print(f'\nCRISPR groups:')
for cat in ['Highly Enriched', 'Moderately Enriched', 'Slightly Enriched', 'Depleted']:
    count = (df[COL_CRISPR_CATEGORY] == cat).sum()
    print(f'  {cat}: {count} ({count/len(df)*100:.1f}%)')

print(f'\nRNA-seq differential expression:')
for status in ['Up', 'Down', 'No_change']:
    count = (df[COL_RNA_STATUS] == status).sum()
    print(f'  {status}: {count} ({count/len(df)*100:.1f}%)')

print(f'\nKey combinations:')
print(f'  CRISPR Highly Enriched + RNA Up: {len(group1)} genes')
print(f'  CRISPR Highly Enriched + RNA Down: {len(group2)} genes')
print(f'  CRISPR Depleted + RNA Up: {len(group3)} genes')
print(f'  CRISPR Depleted + RNA Down: {len(group4)} genes')

print('\n✓ Analysis complete!')
